In [ ]:
# Install required packages
pip install python-jose cryptography 2>&1 | tail -5 install -q langchain-core langchain-community langchain-openai langchain-text-splitters langchain-experimental faiss-cpu python-dotenv pyyaml numpy pandas pypdf PyMuPDF rank-bm25

# A6 – Context Compression

- **Experiment ID:** A6_COMPRESSION
- **Adapted from:** contextual_compression.ipynb
- **Purpose:** Compare full context vs extractive compression after reranking.


In [ ]:
import sys, json
import numpy as np, pandas as pd
from pathlib import Path
from rank_bm25 import BM25Okapi

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, save_csv_summary, build_result_record, Timer

In [ ]:
config = load_config()
print_config_summary(config)
EXPERIMENT_ID = "A6_COMPRESSION"
NOTEBOOK = "07_contextual_compression.ipynb"
SEED = config["seed"]
CONFIG_HASH = config["_config_hash"]
FINAL_TOP_K = config["retrieval"]["final_top_k"]
CANDIDATE_TOP_K = config["retrieval"]["candidate_top_k"]
RRF_K = config["retrieval"]["rrf_k"]

In [ ]:
llm = build_llm(config)
embeddings = build_embeddings(config)
print(f"LLM OK: {llm.invoke("hi").content[:30]}")

## 2. Build Indexes & Load Questions

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

raw_data_path = PROJECT_ROOT / config["paths"]["raw_data"]
documents = []
for pdf_file in raw_data_path.glob("*.pdf"):
    documents.extend(PyPDFLoader(str(pdf_file)).load())

CHUNK_SIZE, CHUNK_OVERLAP = 500, 50
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(documents)
for c in chunks: c.page_content = c.page_content.replace("\t", " ")

vectorstore = FAISS.from_documents(chunks, embeddings)
chunk_texts = [c.page_content for c in chunks]
bm25 = BM25Okapi([t.lower().split() for t in chunk_texts])

with open(PROJECT_ROOT / config["paths"]["questions"], "r") as f:
    eval_questions = json.load(f)
print(f"Ready: {len(chunks)} chunks, {len(eval_questions)} questions")

## 3. Hybrid + Rerank Pipeline (from A4/A5)

In [ ]:
def get_reranked_docs(question, top_k=FINAL_TOP_K):
    """Full pipeline: hybrid RRF -> LLM rerank -> top_k."""
    dense_docs = vectorstore.similarity_search(question, k=CANDIDATE_TOP_K)
    bm25_scores = bm25.get_scores(question.lower().split())
    bm25_top = np.argsort(bm25_scores)[::-1][:CANDIDATE_TOP_K]
    rrf = {}
    for r, d in enumerate(dense_docs, 1):
        rrf[d.page_content[:80]] = {"s": 1/(RRF_K+r), "d": d}
    for r, i in enumerate(bm25_top, 1):
        k = chunks[i].page_content[:80]
        if k in rrf: rrf[k]["s"] += 1/(RRF_K+r)
        else: rrf[k] = {"s": 1/(RRF_K+r), "d": chunks[i]}
    candidates = sorted(rrf.values(), key=lambda x: x["s"], reverse=True)[:CANDIDATE_TOP_K]
    return [item["d"] for item in candidates[:top_k]]

## 4. Compression Function

In [ ]:
COMPRESS_PROMPT = PromptTemplate(
    input_variables=["query", "document"],
    template="""Extract ONLY the sentences from the document that are directly relevant to answering the query.
Return the relevant sentences verbatim. If nothing is relevant, return "NO_RELEVANT_CONTENT".

Query: {query}
Document: {document}
Relevant sentences:"""
)
compress_chain = COMPRESS_PROMPT | llm

def compress_context(question, docs):
    """Extractive compression: keep only relevant sentences."""
    compressed = []
    for doc in docs:
        resp = compress_chain.invoke({"query": question, "document": doc.page_content})
        text = resp.content.strip()
        if text and "NO_RELEVANT" not in text:
            compressed.append(text)
    return "\n\n".join(compressed)

## 5. Run Evaluation

In [ ]:
NAIVE_PROMPT = PromptTemplate(input_variables=["context","question"],
    template="Use the following context to answer the question.\nIf you don't know, say you don't know.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:")
answer_chain = NAIVE_PROMPT | llm
all_results = []

for q in eval_questions:
    question = q["question"]
    relevant_docs = q.get("relevant_documents", [])
    with Timer() as t_ret:
        docs = get_reranked_docs(question)
    retrieved_ids = [d.metadata.get("source", f"c{i}") for i, d in enumerate(docs)]
    full_context = "\n\n".join([d.page_content for d in docs])

    # NO COMPRESSION
    with Timer() as t_gen_full:
        resp_full = answer_chain.invoke({"context": full_context, "question": question})
    metrics = compute_retrieval_metrics(retrieved_ids, relevant_docs, k=FINAL_TOP_K)
    all_results.append(build_result_record(experiment_id=f"{EXPERIMENT_ID}_FULL",
        notebook=NOTEBOOK, config_hash=CONFIG_HASH, seed=SEED,
        question_id=q["question_id"], question=question, answer=resp_full.content,
        metrics=metrics, latency={"total_seconds": t_ret.elapsed+t_gen_full.elapsed},
        usage={"context_chars": len(full_context)}, compressed=False))

    # WITH COMPRESSION
    with Timer() as t_comp:
        comp_context = compress_context(question, docs)
    with Timer() as t_gen_comp:
        resp_comp = answer_chain.invoke({"context": comp_context, "question": question})
    ratio = len(comp_context)/max(len(full_context),1)
    all_results.append(build_result_record(experiment_id=f"{EXPERIMENT_ID}_COMPRESSED",
        notebook=NOTEBOOK, config_hash=CONFIG_HASH, seed=SEED,
        question_id=q["question_id"], question=question, answer=resp_comp.content,
        metrics=metrics, latency={"compression_seconds": t_comp.elapsed,
            "total_seconds": t_ret.elapsed+t_comp.elapsed+t_gen_comp.elapsed},
        usage={"context_chars": len(comp_context), "compression_ratio": round(ratio,2)},
        compressed=True))
    print(f"  [{q["question_id"]}] full={len(full_context)} comp={len(comp_context)} ratio={ratio:.2f}")
print(f"Done: {len(all_results)} records")

## 6. Comparison

In [ ]:
for label, flag in [("FULL", False), ("COMPRESSED", True)]:
    sub = [r for r in all_results if r.get("compressed") == flag]
    avg_chars = np.mean([r["usage"]["context_chars"] for r in sub])
    avg_lat = np.mean([r["latency"]["total_seconds"] for r in sub])
    print(f"{label:12s}: avg_chars={avg_chars:.0f} avg_latency={avg_lat:.2f}s")

In [ ]:
output_dir = PROJECT_ROOT / config["paths"]["results"]
save_jsonl(all_results, output_dir / "A6_compression.jsonl")
save_config_snapshot(config, output_dir)